In [ ]:
%pip install -q gradio transformers accelerate qwen-vl-utils pillow
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00


In [ ]:
%pip install -q -U transformers accelerate gradio qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.3 MB/s eta 0:00:00


In [ ]:
import gc
import time
import torch
import gradio as gr

from PIL import Image
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    Qwen2_5_VLForConditionalGeneration
)

from qwen_vl_utils import process_vision_info

print("Transformers imports successful!")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODELS = {
    "Qwen2.5-VL-3B": {
        "id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "type": "qwen"
    },
    "Qwen2.5-VL-7B": {
        "id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "type": "qwen"
    },
    "SmolVLM-500M": {
        "id": "HuggingFaceTB/SmolVLM-500M-Instruct",
        "type": "smol"
    },
    "InternVL2.5-4B": {
        "id": "OpenGVLab/InternVL2_5-4B",
        "type": "internvl"
    }
}

loaded_model = None
loaded_processor = None
loaded_model_name = None

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: No GPU detected. Vision-language inference may be slow.")

Transformers imports successful!
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA available: True
GPU: Tesla T4


In [ ]:
def unload_model():
    global loaded_model, loaded_processor, loaded_model_name

    if loaded_model is not None:
        del loaded_model

    if loaded_processor is not None:
        del loaded_processor

    loaded_model = None
    loaded_processor = None
    loaded_model_name = None

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_selected_model(model_name):
    global loaded_model, loaded_processor, loaded_model_name

    if model_name not in MODELS:
        return f"Unknown model: {model_name}"

    if loaded_model_name == model_name and loaded_model is not None:
        return f"{model_name} is already loaded."

    unload_model()

    model_info = MODELS[model_name]
    model_id = model_info["id"]
    model_type = model_info["type"]

    try:
        print(f"Loading {model_name}...")

        if model_type == "internvl":
            return (
                "InternVL2.5-4B is not enabled yet. "
                "It requires a dedicated model-loading and inference implementation."
            )

        loaded_processor = AutoProcessor.from_pretrained(model_id)

        if model_type == "qwen":
            loaded_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                model_id,
                torch_dtype="auto",
                device_map="auto"
            )

        elif model_type == "smol":
            loaded_model = AutoModelForImageTextToText.from_pretrained(
                model_id,
                torch_dtype="auto",
                device_map="auto"
            )

        else:
            unload_model()
            return f"Unsupported model type: {model_type}"

        loaded_model.eval()
        loaded_model_name = model_name

        return f"Successfully loaded {model_name} ({model_id})."

    except Exception as e:
        unload_model()
        return f"Model loading failed: {type(e).__name__}: {str(e)[:1000]}"

In [ ]:
def answer_question(before_image, after_image, question, max_new_tokens):
    if before_image is None:
        return "Please upload the BEFORE satellite image.", ""

    if after_image is None:
        return "Please upload the AFTER satellite image.", ""

    if not question or not question.strip():
        return "Please enter a question.", ""

    if loaded_model is None:
        return "Please select a model and click Load / Switch Model.", ""

    before_image = before_image.convert("RGB")
    after_image = after_image.convert("RGB")

    prompt = (
        "You are analyzing satellite images for change detection. "
        "The first image is BEFORE and the second image is AFTER. "
        "Compare the two images carefully and answer the question using "
        "visual evidence. If the evidence is insufficient, state that clearly.\n\n"
        f"Question: {question.strip()}"
    )

    try:
        start_time = time.perf_counter()

        if MODELS[loaded_model_name]["type"] == "qwen":
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "BEFORE satellite image:"},
                        {"type": "image", "image": before_image},
                        {"type": "text", "text": "AFTER satellite image:"},
                        {"type": "image", "image": after_image},
                        {"type": "text", "text": prompt}
                    ]
                }
            ]

            chat_prompt = loaded_processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            image_inputs, video_inputs = process_vision_info(messages)

            inputs = loaded_processor(
                text=[chat_prompt],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt"
            )

            inputs = inputs.to(loaded_model.device)

            with torch.inference_mode():
                generated_ids = loaded_model.generate(
                    **inputs,
                    max_new_tokens=int(max_new_tokens),
                    do_sample=False
                )

            generated_ids = generated_ids[
                :, inputs.input_ids.shape[1]:
            ]

            answer = loaded_processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

        elif MODELS[loaded_model_name]["type"] == "smol":
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": before_image},
                        {"type": "image", "image": after_image},
                        {"type": "text", "text": prompt}
                    ]
                }
            ]

            inputs = loaded_processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            )

            inputs = inputs.to(loaded_model.device)

            with torch.inference_mode():
                generated_ids = loaded_model.generate(
                    **inputs,
                    max_new_tokens=int(max_new_tokens),
                    do_sample=False
                )

            generated_ids = generated_ids[
                :, inputs["input_ids"].shape[1]:
            ]

            answer = loaded_processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

        else:
            answer = (
                "InternVL2.5-4B requires its model-specific image preprocessing "
                "and generation interface. Use the dedicated InternVL adapter "
                "before evaluating this model."
            )

        elapsed = time.perf_counter() - start_time

        return answer, f"{elapsed:.2f} seconds"

    except Exception as e:
        return (
            f"Inference failed: {type(e).__name__}: {str(e)[:1000]}",
            ""
        )

In [ ]:
with gr.Blocks(title="Satellite Image Change Understanding") as demo:

    gr.Markdown(
        """
        # Satellite Image Change Understanding

        Upload BEFORE and AFTER satellite images, select a vision-language
        model, load it, and ask a question about the changes.

        Only one model is loaded at a time.
        """
    )

    with gr.Row():
        model_selector = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="Qwen2.5-VL-3B",
            label="Select Model"
        )

        load_button = gr.Button(
            "Load / Switch Model",
            variant="primary"
        )

    model_status = gr.Textbox(
        label="Model Status",
        interactive=False
    )

    with gr.Row():
        before_input = gr.Image(
            type="pil",
            label="BEFORE Satellite Image"
        )

        after_input = gr.Image(
            type="pil",
            label="AFTER Satellite Image"
        )

    question_input = gr.Textbox(
        label="Your Question",
        placeholder="What changes occurred between the two satellite images?",
        lines=3
    )

    max_tokens = gr.Slider(
        minimum=32,
        maximum=512,
        value=128,
        step=32,
        label="Maximum New Tokens"
    )

    analyze_button = gr.Button(
        "Analyze Change",
        variant="primary"
    )

    answer_output = gr.Textbox(
        label="Model Answer",
        lines=8
    )

    response_time_output = gr.Textbox(
        label="Inference Time",
        interactive=False
    )

    load_button.click(
        fn=load_selected_model,
        inputs=[model_selector],
        outputs=[model_status]
    )

    analyze_button.click(
        fn=answer_question,
        inputs=[
            before_input,
            after_input,
            question_input,
            max_tokens
        ],
        outputs=[
            answer_output,
            response_time_output
        ]
    )

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c616d03494e91eda1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading Qwen2.5-VL-3B...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loading Qwen2.5-VL-7B...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]